In [30]:
import pandas as pd

# Load original data
df = pd.read_csv("/Users/anisa/OneDrive/Desktop/fyp/notebooks/datab.csv")
df["date"] = pd.to_datetime(df["date"])

# Load backtest forecasts
direct = pd.read_csv("/Users/anisa/OneDrive/Desktop/fyp/data/cleaned/tcn/tcn_direct_forecasts (1).csv")
recursive = pd.read_csv("/Users/anisa/OneDrive/Desktop/fyp/data/cleaned/tcn/tcn_recursive_forecasts.csv")

# Load metrics
metrics_direct = pd.read_csv("/Users/anisa/OneDrive/Desktop/fyp/data/cleaned/tcn/tcn_direct_metrics.csv", index_col=0)
metrics_direct = direct.reset_index().rename(columns={"index": "series_id"})
metrics_direct.to_csv("tcn_direct_metrics_fixed.csv", index=False)
metrics_recursive = pd.read_csv("/Users/anisa/OneDrive/Desktop/fyp/data/cleaned/tcn/tcn_recursive_metrics.csv", index_col=0)
metrics_recursive = recursive.reset_index().rename(columns={"index": "series_id"})
metrics_recursive.to_csv("tcn_recursive_metrics_fixed.csv", index=False)
direct.head(), recursive.head()


(         time  resistance_pct                        series_id
 0  2023-06-01       10.716024  E-coli__Amikacin__East Midlands
 1  2023-12-01       12.770133  E-coli__Amikacin__East Midlands
 2  2024-06-01       12.072960  E-coli__Amikacin__East Midlands
 3  2024-12-01       11.695554  E-coli__Amikacin__East Midlands
 4  2025-06-01       13.617923  E-coli__Amikacin__East Midlands,
          time  resistance_pct                        series_id
 0  2023-01-01       10.713719  E-coli__Amikacin__East Midlands
 1  2023-02-01       11.612662  E-coli__Amikacin__East Midlands
 2  2023-03-01       12.369230  E-coli__Amikacin__East Midlands
 3  2023-04-01       13.269888  E-coli__Amikacin__East Midlands
 4  2023-05-01       13.327518  E-coli__Amikacin__East Midlands)

In [31]:
df.head()

,Unnamed: 0,geography,stratum,date,organism,resistance_pct,series_id,model_region,prescribing_rate
0,0,East of England,Amikacin,2020-08-01,K-pneumoniae,0.64,K-pneumoniae__Amikacin__East of England,East of England,1.52
1,1,East of England,Carbapenems,2020-08-01,K-pneumoniae,0.00,K-pneumoniae__Carbapenems__East of England,East of England,1.52
2,2,East of England,Ciprofloxacin,2020-08-01,K-pneumoniae,11.06,K-pneumoniae__Ciprofloxacin__East of England,East of England,1.52
3,3,East of England,Co-amoxiclav,2020-08-01,K-pneumoniae,27.04,K-pneumoniae__Co-amoxiclav__East of England,East of England,1.52
4,4,East of England,Gentamicin,2020-08-01,K-pneumoniae,4.90,K-pneumoniae__Gentamicin__East of England,East of England,1.52


In [32]:
direct = pd.read_csv("tcn_direct_metrics_fixed.csv")
direct.head()


,series_id,time,resistance_pct,series_id.1
0,0,2023-06-01,10.716024,E-coli__Amikacin__East Midlands
1,1,2023-12-01,12.770133,E-coli__Amikacin__East Midlands
2,2,2024-06-01,12.072960,E-coli__Amikacin__East Midlands
3,3,2024-12-01,11.695554,E-coli__Amikacin__East Midlands
4,4,2025-06-01,13.617923,E-coli__Amikacin__East Midlands


In [33]:
recursive = pd.read_csv("tcn_recursive_metrics_fixed.csv")
recursive.head()

,series_id,time,resistance_pct,series_id.1
0,0,2023-01-01,10.713719,E-coli__Amikacin__East Midlands
1,1,2023-02-01,11.612662,E-coli__Amikacin__East Midlands
2,2,2023-03-01,12.369230,E-coli__Amikacin__East Midlands
3,3,2023-04-01,13.269888,E-coli__Amikacin__East Midlands
4,4,2023-05-01,13.327518,E-coli__Amikacin__East Midlands


In [136]:
from darts import TimeSeries
from darts.models import TCNModel
from darts.dataprocessing.transformers import Scaler
import pandas as pd

# 1. Rebuild TimeSeries objects from df
target_series = {}
feature_series = {}

for sid, group in df.groupby("series_id"):
    group = group.sort_values("date")

    target = TimeSeries.from_dataframe(
        group, time_col="date", value_cols="resistance_pct", freq="MS"
    )
    feature = TimeSeries.from_dataframe(
        group, time_col="date", value_cols="prescribing_rate", freq="MS"
    )

    target_series[sid] = target
    feature_series[sid] = feature

# 2. Refit scalers





In [137]:
tcn_target_scaled = {}
tcn_cov_scaled = {}
tcn_target_scalers = {}
tcn_cov_scalers = {}

for sid in target_series.keys():
     ts = target_series[sid]
     cov = feature_series[sid]
     t_scaler = Scaler()
     c_scaler = Scaler()
     tcn_target_scaled[sid] = t_scaler.fit_transform(ts)
     tcn_cov_scaled[sid] = c_scaler.fit_transform(cov)
     tcn_target_scalers[sid] = t_scaler
     tcn_cov_scalers[sid] = c_scaler


In [138]:
# 3. Build recursive TCN
def build_tcn_recursive():
    return TCNModel(
        input_chunk_length=12,
        output_chunk_length=1,
        n_epochs=10,
        dropout=0.2,
        kernel_size=3,
        dilation_base=2,
        weight_norm=True,
        random_state=42
    )


In [139]:
# 4. Predict next 6 months
def train_tcn_final_models(target_scaled, cov_scaled):
    models = {}

    for sid in target_scaled.keys():
        ts  = target_scaled[sid]
        cov = cov_scaled[sid]

        model = build_tcn_recursive()
        model.fit(ts, past_covariates=cov, verbose=False)
        models[sid] = model

    return models

tcn_final_models = train_tcn_final_models(tcn_target_scaled, tcn_cov_scaled)


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


c:\Users\anisa\miniconda3\envs\darts310\lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: FutureWarning:

`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

c:\Users\anisa\miniconda3\envs\darts310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning:

'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.

`Trainer.fit` stopped: `max_epochs=10` reached.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\Users\anisa\miniconda3\envs\darts310\lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: FutureWarning:

`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

c:\Users\anisa\miniconda3\envs\darts310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning:

'pin_memory' argument is set as true but no acc

In [142]:
def build_baseline_future_covariates(feature_series, horizon=6):
    future_covs = {}

    for sid, cov_ts in feature_series.items():
        last_val = cov_ts.values()[-1][0]

        future_dates = pd.date_range(
            start=cov_ts.end_time() + cov_ts.freq,
            periods=horizon,
            freq=cov_ts.freq
        )

        future_covs[sid] = TimeSeries.from_times_and_values(
            future_dates,
            [last_val] * horizon
        )

    return future_covs


In [143]:
tcn_baseline_future_covs = build_baseline_future_covariates(feature_series, horizon=6)


In [144]:
from darts import TimeSeries
import pandas as pd

def forecast_6m_tcn(tcn_models, tcn_target_scalers, tcn_cov_scaled, tcn_baseline_future_covs, tcn_cov_scalers):
    forecasts = {}

    for sid, model in tcn_models.items():
        cov_scaled = tcn_cov_scaled[sid]
        future_cov = tcn_baseline_future_covs[sid]

        c_scaler = tcn_cov_scalers[sid]
        t_scaler = tcn_target_scalers[sid]

        future_cov_scaled = c_scaler.transform(future_cov)
        full_cov_scaled = cov_scaled.append(future_cov_scaled)

        fc_scaled = model.predict(
            n=6,
            past_covariates=full_cov_scaled
        )

        fc = t_scaler.inverse_transform(fc_scaled)
        forecasts[sid] = fc

    return forecasts


In [145]:
tcn_final_6m = forecast_6m_tcn( 
    tcn_final_models,
    tcn_target_scalers,
    tcn_cov_scaled,
    tcn_baseline_future_covs,
    tcn_cov_scalers
      )

`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\Users\anisa\miniconda3\envs\darts310\lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: FutureWarning:

`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

c:\Users\anisa\miniconda3\envs\darts310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning:

'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.



Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00,  7.87it/s]

`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\Users\anisa\miniconda3\envs\darts310\lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: FutureWarning:

`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

c:\Users\anisa\miniconda3\envs\darts310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning:

'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.



Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 14.09it/s]

...

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\Users\anisa\miniconda3\envs\darts310\lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: FutureWarning:

`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

c:\Users\anisa\miniconda3\envs\darts310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning:

'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.



Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 58.82it/s]

`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\Users\anisa\miniconda3\envs\darts310\lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: FutureWarning:

`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

c:\Users\anisa\miniconda3\envs\darts310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning:

'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.



Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 52.63it/s]


In [146]:
def forecasts_to_df(forecast_dict): 
    rows = [] 
    for sid, ts in forecast_dict.items(): 
        df_fc = ts.to_dataframe().reset_index() 
        df_fc["series_id"] = sid 
        df_fc.rename(columns={"index": "date", "resistance_pct": "value"}, inplace=True) 
        rows.append(df_fc) 
    return pd.concat(rows, ignore_index=True) 

df_future = forecasts_to_df(tcn_final_6m) 
df_future.rename(columns={"time": "date"}, inplace=True)

In [147]:
df_actual = df[["date", "resistance_pct", "series_id"]].copy() 
df_actual.rename(columns={"resistance_pct": "value"}, inplace=True) 
df_future["is_forecast"] = True 
df_actual["is_forecast"] = False 
df_all = pd.concat([df_actual, df_future], ignore_index=True) 
df_all = df_all.sort_values(["series_id", "date"]) 
df_all["date"] = pd.to_datetime(df_all["date"])

In [148]:
import plotly.graph_objects as go
def plot_continuous(df_all, sid):
    df_sid = df_all[df_all["series_id"] == sid].sort_values("date")

    actual = df_sid[df_sid["is_forecast"] == False]
    forecast = df_sid[df_sid["is_forecast"] == True]

    # Anchor forecast to last actual point
    if not actual.empty and not forecast.empty:
        last_actual_point = actual.tail(1)
        forecast = pd.concat([last_actual_point, forecast])

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=actual["date"],
        y=actual["value"],
        mode="lines",
        name="Actual",
        line=dict(color="blue")
    ))

    fig.add_trace(go.Scatter(
        x=forecast["date"],
        y=forecast["value"],
        mode="lines",
        name="Forecast",
        line=dict(color="red")
    ))

    fig.update_layout(
        title=f"{sid} — Continuous TCN Forecast",
        xaxis_title="Date",
        yaxis_title="Resistance (%)",
        template="plotly_white",
        hovermode="x unified"
    )

    fig.show()

# Example usage
plot_continuous(df_all, "e-faecium__Glycopeptides__South East")


In [150]:
df_actuals = df[["date", "series_id", "resistance_pct"]].copy()
df_actuals.rename(columns={"resistance_pct": "value"}, inplace=True)
df_actuals.to_csv("/Users/anisa/OneDrive/Desktop/fyp/data/actuals.csv", index=False)

In [151]:
# === EXPORT TCN BASELINE CONTINUOUS FORECAST ===

# df_all already contains actual + forecast for ALL series
# but we need to add model="tcn"

df_all_tcn = df_all.copy()
df_all_tcn["model"] = "tcn"

df_all_tcn.to_csv("/Users/anisa/OneDrive/Desktop/fyp/data/tcn_continuous.csv", index=False)

print("Saved: tcn_continuous.csv")



Saved: tcn_continuous.csv
